# NLLB-200 on Sicilian — zero-shot baseline

Evaluates Meta's **NLLB-200** on our Arba-Sicula test set, both directions, with BLEU + chrF (same metrics as the local Sockeye baseline). NLLB supports Sicilian as `scn_Latn`.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough for the 600M model).
Then run the cells top to bottom and, when prompted, upload `test.scn` and `test.en` from `data/dataset/`.

In [ ]:
!pip -q install transformers sentencepiece sacrebleu

In [ ]:
# Upload test.scn and test.en (from data/dataset/) when prompted
from google.colab import files
up = files.upload()
scn = open('test.scn', encoding='utf-8').read().splitlines()
en  = open('test.en',  encoding='utf-8').read().splitlines()
print(len(scn), 'scn lines |', len(en), 'en lines')

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL = 'facebook/nllb-200-distilled-600M'   # or facebook/nllb-200-1.3B
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()
print('loaded', MODEL, 'on', device)

In [ ]:
def translate(texts, src_lang, tgt_lang, bs=32, max_len=160):
    tok.src_lang = src_lang
    tgt_id = tok.convert_tokens_to_ids(tgt_lang)
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True,
                  truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id,
                                  max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  {min(i+bs,len(texts))}/{len(texts)}', end='\r')
    return out

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

def report(hyp, ref, tag):
    b = BLEU().corpus_score(hyp, [ref]).score
    c = CHRF().corpus_score(hyp, [ref]).score
    print(f'{tag}:  BLEU={b:.2f}  chrF={c:.2f}')

print('scn->en ...'); h1 = translate(scn, 'scn_Latn', 'eng_Latn'); report(h1, en, 'NLLB scn->en')
print('en->scn ...'); h2 = translate(en, 'eng_Latn', 'scn_Latn'); report(h2, scn, 'NLLB en->scn')

## Next: fine-tuning

If zero-shot looks promising, fine-tune on `train.scn`/`train.en` (upload them too). Options: full fine-tune of the 600M model on a T4, or LoRA (peft) for the 1.3B/3.3B. Keep the same test set so numbers stay comparable. Ask the assistant to add the fine-tuning cell.